### police arrests demo to understand how polarization works

In [1]:
import jax
import jax.numpy as jnp
from memo import memo
from memo import domain as product
from enum import IntEnum
from matplotlib import pyplot as plt
from jax.scipy.stats.norm import pdf as normpdf

In [2]:

normpdfjit = jax.jit(normpdf)

PolicePerformance = jnp.linspace(0, 1, 10+1, endpoint=True)

Causal = jnp.linspace(-1, 1, 40+1, endpoint=True)

Arrests = jnp.linspace(0, 1, 10+1, endpoint=True)


In [4]:
import jax
import jax.numpy as jnp
from memo import memo
from memo import domain as product
from enum import IntEnum
from matplotlib import pyplot as plt
from jax.scipy.stats.norm import pdf as normpdf

normpdfjit = jax.jit(normpdf)

PolicePerformance = jnp.linspace(0, 1, 10+1, endpoint=True)

Causal = jnp.linspace(-1, 1, 40+1, endpoint=True)

Arrests = jnp.linspace(0, 1, 10+1, endpoint=True)

@jax.jit
def arrests_pdf(arrests, performance, causal_link):
    arrests_mu = causal_link * (performance - 0.5) + 0.5
    arrests_sigma = 0.1
    return normpdf(arrests, arrests_mu, arrests_sigma)

@jax.jit
def reported_cl_pdf(reported, real, bias=0.0):
    return normpdf(reported, real + real*bias, 2.0)

@memo
def viewerModel[
    _prior_expectation_cl: Causal,
](
    reported_cl_observed, 
    nobs,
):
    viewer: knows(
        _prior_expectation_cl,
    )
    viewer: thinks[
        police: given(causal_link in Causal, wpp=(
            viewerModel[causal_link](reported_cl_observed, nobs - 1) 
            if nobs > 0 else 1)),
        police: chooses(performance in PolicePerformance, wpp=1),
        police: chooses(arrests in Arrests, wpp=arrests_pdf(arrests, performance, causal_link)),
        news: knows(police.causal_link),
        news: chooses(reported_cl in Causal, wpp=(
            reported_cl_pdf(reported_cl, police.causal_link)
        )),
    ]
    # viewer: observes[news.reported_cl] is _prior_expectation_cl
    viewer: observes_event(wpp=normpdfjit(news.reported_cl, reported_cl_observed, 0.2))
    # viewer: observes_that(news.reported_cl > reported_cl_observed )
    
    
    return viewer[ 
        Pr[
            police.causal_link == _prior_expectation_cl
        ] 
    ]

In [5]:
def viewerModel[
    _prior_expectation_cl: Causal,](
    reported_cl_observed, 
    nobs, #number of observations
):
    viewer: knows(
        _prior_expectation_cl,
    )
    viewer: thinks[ #how is the causal_link established?
        police: given(causal_link in Causal, wpp = (
            viewerModel(causal_link, reported_cl_observed, nobs-1)  # why is it nob-1?
        if nobs>0 else 1)),
        police: chooses(performance in PolicePerformance, wpp=1),
        police: chooses(arrests in Arrests, wpp=arrests_pdf(arrests, performance, causal_link)),
        news: knows(police.causal_link), 
        police: knows(police.causal_link),
        news: chooses(reported_cl in Causal, wpp=reported_cl_pdf(reported_cl, police.causal_link)),
    
    ]
    
    viewer: observes_event(wpp= normpdfjit(news.reported_cl, reported_cl_observed, 0.2))
    
    
    return viewer[
        Pr[
            police.causal_link == _prior_expectation_cl
        ]
    ]

In [ ]:
from memo import memo
import jax
import jax.numpy as np
# from icecream import ic
from functools import cache
from enum import IntEnum

# states represent the true severity of patient's condition
class S(IntEnum):
    Severe = 0
    Moderate = 1
    Mild = 2

# actions represent different treatment options
class A(IntEnum):
    IntensiveTreatment = 0
    ModerateTreatment = 1
    Observation = 2

# observations are symptoms and pain report
class O(IntEnum):
    HighSymptoms = 0
    ModerateSymptoms = 1
    LowSymptoms = 2

# belief space, discretized into 50 possible belief states
# represents P(Severe)
B = np.linspace(0, 1, 50)


@jax.jit
def get_belief(b, s):
    # create a vector of probabilities for each state
    probs = np.zeros(3)
    probs = probs.at[S.Severe].set(b)
    probs = probs.at[S.Moderate].set((1 - b) * 0.6)
    probs = probs.at[S.Mild].set((1 - b) * 0.4)
    return probs[s]


@jax.jit
def Tr(s, a, s_):
    # transition probabilities: P(s_|s,a)
    # how the patient's condition changes based on treatment
    transitions = np.array([
        # P(s_|Severe, a)
        [
            [0.2, 0.5, 0.3],  # IntensiveTreatment: good chance of improvement
            [0.6, 0.3, 0.1],  # ModerateTreatment: likely stays severe
            [0.9, 0.1, 0.0]   # Observation: likely worsens
        ],
        # P(s_|Moderate, a)
        [
            [0.1, 0.3, 0.6],  # IntensiveTreatment: good chance of improvement
            [0.2, 0.5, 0.3],  # ModerateTreatment: balanced outcomes, still good chance of improvement
            [0.3, 0.6, 0.1]   # Observation: might worsen
        ],
        # P(s_|Mild, a)
        [
            [0.0, 0.2, 0.8],  # IntensiveTreatment: stays mild or moderate (side effects)
            [0.0, 0.1, 0.9],  # ModerateTreatment: likely stays mild
            [0.1, 0.2, 0.7]   # Observation: mostly stays mild, might worsen
        ]
    ])
    return transitions[s, a, s_]


@jax.jit
def Obs(o, s, a):
    # observation probabilities: P(o|s,a)
    # what symptoms we observe given the true state and treatment
    observations = np.array([
        
        # P(o|Severe, a)
        [
            [0.8, 0.2, 0.0],  # IntensiveTreatment: symptoms mostly high
            [0.7, 0.2, 0.1],  # ModerateTreatment: symptoms mostly high
            [0.9, 0.1, 0.0]   # Observation: symptoms clearly high
        ],
        # P(o|Moderate, a)
        [
            [0.3, 0.6, 0.1],  # IntensiveTreatment: symptoms mostly moderate
            [0.2, 0.7, 0.1],  # ModerateTreatment: symptoms mostly moderate
            [0.3, 0.6, 0.1]   # Observation: symptoms mostly moderate
        ],
        # P(o|Mild, a)
        [
            [0.1, 0.3, 0.6],  # IntensiveTreatment: symptoms mostly low
            [0.0, 0.2, 0.8],  # ModerateTreatment: symptoms mostly low
            [0.0, 0.1, 0.9]   # Observation: symptoms clearly low
        ]
    ])
    return observations[s, a, o]

@jax.jit
def R_doctor(s, a):
    # reward function balances health outcomes with treatment costs/risks
    
    # health state costs (being in worse states is worse)
    health_cost = np.array([-30, -12, -3])[s]
    
    # treatment costs and risks
    monetary_gain = np.array([
        10,  # IntensiveTreatment: staying at the hospital
        0,   # ModerateTreatment: milder side effects, otc medication
        3    # Observation: another appointment
    ])[a]
    
    # reponse preference
    communication_cost = np.array([
        -3,  # IntensiveTreatment: staying at the hospital
        2,   # ModerateTreatment: milder side effects, otc medication
        -1    # Observation: another appointment
    ])[a] # we can make this conditional // if the person is actually very sick they would not like getting underdiagnosed
    # but if they are moderately or mildly sick they would like to not freak out 
    
    
    # this might not be necessary as it is already implicit in the model but worth trying twice perhaps
    # # additional rewards for appropriate treatment matching
    # # intensive treatment is good for severe, bad for mild (side effects)
    # # moderate treatment is good for moderate
    # # observation is good for mild, bad for severe (missed opportunity)
    treatment_match = np.array([
        [5, -5, -10],   # Severe: intensive good, observation bad
        [-2, 3, -3],    # Moderate: moderate treatment best
        [-8, 0, 2]      # Mild: intensive bad (side effects), observation fine 
    ])[s, a]
    
    return health_cost + monetary_gain + treatment_match




@jax.jit
def R_patient(s, a):
    # reward function balances health outcomes with treatment costs/risks
    
    # health state costs (being in worse states is worse)
    health_cost = np.array([-30, -12, -3])[s]
    
    # treatment costs and risks
    treatment_cost = np.array([
        -15,  # IntensiveTreatment: expensive, side effects
        -8,   # ModerateTreatment: moderate cost
        -1    # Observation: minimal cost but no direct benefit
    ])[a]
    
    # this might not be necessary as it is already implicit in the model but worth trying twice perhaps
    # # additional rewards for appropriate treatment matching
    # # intensive treatment is good for severe, bad for mild (side effects)
    # # moderate treatment is good for moderate
    # # observation is good for mild, bad for severe (missed opportunity)
    # treatment_match = np.array([
    #     [5, -5, -10],   # Severe: intensive good, observation bad
    #     [-2, 3, -3],    # Moderate: moderate treatment best
    #     [-8, 0, 2]      # Mild: intensive bad (side effects), observation fine
    # ])[s, a]
    
    return health_cost + treatment_cost



In [ ]:
@jax.jit
def R_doctor(s, a):
    # base health impact (objective medical outcome)
    health_impact = np.array([-30, -12, -3])[s]
    
    # financial incentives (remain constant)
    financial_incentive = np.array([0, 5, 7])[a]
    

    
    # # risk aversion varies by gender (higher for male patients)
    # risk_aversion = np.array([1.2, 0.8])[gender]
    
    # # treatment appropriateness matrix (base values)
    # treatment_match_base = np.array([
    #     [5, -5, -10],   # severe: intensive good, observation bad
    #     [-2, 3, -3],    # moderate: moderate treatment best
    #     [-8, 0, 2]      # mild: intensive bad, observation good
    # ])[s, a]
    
    # # apply perception bias to treatment matching
    # treatment_match = treatment_match_base 
    
    # # communication/validation component
    # validation_value = np.array([
    #     # [IntensiveTreatment, ModerateTreatment, Observation]
    #     [-2, 0, -5],  # severe
    #     [0, 1, -2],   # moderate
    #     [2, 0, -1]    # mild
    # ])[s, a]
    
    # # gender-specific validation adjustment
    # validation_factor = np.array([1.0, 1.5])[gender]  # Male=1.0, Female=1.5
    # validation_component = validation_value * validation_factor
    
    # # risk component (penalty for uncertain treatments)
    # uncertainty = np.array([
    #     [0.2, 0.3, 0.1],  # severe: intensive most certain
    #     [0.3, 0.2, 0.3],  # moderate: moderate treatment most certain
    #     [0.4, 0.2, 0.1]   # mild: observation most certain
    # ])[s, a]
    # risk_component = -risk_aversion * uncertainty * 10
    
    # total reward
    return health_impact + financial_incentive 

In [ ]:
## can add multiple rounds


@memo(cache=True)
def Q[b: B, a: A](t):
    doctor: knows(b, a) # belief and action
    doctor: thinks[
        env: knows(b, a),
        env: given(s in S, wpp=get_belief(b, s)),
        env: given(s_ in S, wpp=Tr(s, a, s_)),
        env: given(o in O, wpp=Obs(o, s_, a))
    ]
    doctor: snapshots_self_as(future_doctor)
    
    return doctor[
        E[R_doctor(env.s, a)] + (0.0 if t <= 0 else 0.9 * imagine[
            future_doctor: observes [env.o] is env.o,
            future_doctor: chooses(b_ in B, wpp=exp(-130.0 * abs(E[env.s_ == 0] + E[env.s_ == 1]/2  - b_))), # this statement is changed to maximize mild symptoms
            future_doctor: chooses(a_ in A, to_maximize=Q[b_, a_](t - 1)),
            E[future_doctor[ Q[b_, a_](t - 1) ] ]
        ])
    ]


%timeit -r 10 -n 10 Q.cache_clear(); Q(100).block_until_ready()


Currently running into an issue while trying to get observing the event to work :/

In [6]:
reported = Causal[4]
for nobs_ in range(10):
    res_viewer = viewerModel(reported, nobs_, print_table=False, return_aux=True, return_xarray=True)
    xardata = res_viewer.aux.xarray
    print(f"nobs: {nobs_+1} E = {jnp.dot(xardata['_prior_expectation_cl'].values, xardata.values)}")

TypeError: viewerModel() got an unexpected keyword argument 'print_table'